# 期末作业 [困难]：概念综合：设计 Inception-ResNet 混合模块

### 作业目标：
1.  **核心：** 证明你已深度理解 GoogLeNet 的 `Inception` 模块（多分支并行）。
2.  **核心：** 证明你已深度理解 ResNet 的 `BasicBlock` 模块（残差连接）。
3.  **核心：** 综合上述两个概念，设计并实现一个全新的、你自己的 **`InceptionResBlock`** 模块。
4.  使用你设计的新模块构建一个小型网络，并在 CIFAR-10 数据集上成功训练。

### 作业背景：
在深度学习中，最强大的网络往往是多种优秀设计的结合体。`Inception` 模块解决了多尺度特征提取问题，而 `ResNet` 解决了深度网络的梯度消失问题。将它们结合起来，是构建更强大网络的自然思路。

本次作业要求你不再是“复现”，而是要“设计”和“综合”。

### 教学参考：
本项目需要你同时参考 `第六节` (GoogLeNet) 和 `第七节` (ResNet) 的 `model.py` 脚本。

---

## 1. 准备工作：导入库与设置

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm
import os
import sys

# 设置 device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. 数据准备 (CIFAR-10)

我们将使用 CIFAR-10 (3通道, 32x32) 来测试你的新模块。

In [ ]:
batch_size = 128
learning_rate = 0.001
epochs = 20

# CIFAR-10 均值和标准差
transform_train = transforms.Compose([
    
])

transform_test = transforms.Compose([
  
])

train_dataset = datasets.CIFAR10
test_dataset = datasets.CIFAR10

train_loader = 
test_loader = 

print("Data loaded.")

## 3. [核心] 设计 `InceptionResBlock` 模块

你的任务是实现这个类。它需要结合 `Inception` 和 `BasicBlock` 的特点。

**设计要求：**
1.  **Inception 结构**：模块内部应有多个并行的卷积分支。我们简化一下，只用两个分支：
    * 分支 1: `BasicConv2d` (1x1 卷积)
    * 分支 2: `BasicConv2d` (1x1 卷积) -> `BasicConv2d` (3x3 卷积, padding=1)
2.  **通道拼接**：在 `forward` 中，将这两个分支的输出 `torch.cat` 在一起。
3.  **通道匹配**：拼接后的通道数会变多。你需要添加一个 `nn.Linear` 或 `1x1 Conv` (推荐 `BasicConv2d`)，将拼接后的通道数**降维**回与输入通道数 `in_channels` 一致。
4.  **ResNet 结构**：将降维后的输出与原始输入 `x` 相加 (`out = out + identity`)。
5.  **激活函数**：在相加后，应用 `ReLU` 激活函数。
6.  **下采样**：你需要一个 `stride` 参数。如果 `stride != 1`，则需要对 `identity` 路径（即 `x`）进行下采样（使用一个 `1x1 Conv`），就像 `BasicBlock` 中的 `downsample` 一样。同时，`Inception` 分支中的某个卷积（例如分支2的 `3x3` 卷积）也需要使用 `stride` 来缩小特征图尺寸。

### 3.1. `BasicConv2d` 辅助类

我们直接使用 GoogLeNet 中的 `BasicConv2d` 辅助类，它封装了 `Conv-BN-ReLU`，非常方便。

In [ ]:
class BasicConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, **kwargs):
        super(BasicConv2d, self).__init__()


    def forward(self, x):

        return x

### 3.2. `InceptionResBlock` 实现

请你填充 `__init__` 和 `forward` 方法。

In [ ]:
class InceptionResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(InceptionResBlock, self).__init__()
        
        self.stride = stride
        # 注意：为了简化，我们让 out_channels 代表模块*内部*的通道数，
        # 最终输出的通道数将与 in_channels 保持一致（或通过 downsample 改变）
        
        # TODO: 分支 1 (1x1 卷积)
        self.branch1 = 

        # TODO: 分支 2 (1x1 -> 3x3 卷积)
        self.branch2 = nn.Sequential(
            
        )
        
        # TODO: 通道匹配的 1x1 卷积
        # 两个分支拼接后通道数为 out_channels * 2
        # 我们需要将其降维回 in_channels (如果 stride=1)
        # 或者降维到我们期望的输出通道（例如 out_channels*2? 为了演示，我们统一降回 out_channels*2）
        # 挑战：如何让输出通道匹配 `identity`？
        # 让我们重新定义一下：目标输出通道 = out_channels
        # 那么两个分支的输出通道可以是 out_channels // 2
        
        # --- 让我们采用更简单、更像 ResNet 的设计 ---
        # 目标：in_channels -> out_channels
        
        # 重置：
        # 分支1: 1x1 卷积
        self.b1 = 
        
        # 分支2: 1x1 -> 3x3 卷积
        self.b2 = nn.Sequential(
            
        )
        
        # 拼接后的通道数 = (out_channels // 2) + (out_channels // 2) = out_channels
        # 这样就不需要额外的1x1卷积来匹配通道了！
        
        # TODO: 定义 `downsample` 路径 (参考 ResNet)
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = BasicConv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        
        # TODO: 定义 `stride` 应用在哪
        # 我们让两个分支都负责下采样
        self.b1 = 
        self.b2_s1 = 
        self.b2_s2 = 
        
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # TODO: 1. 定义 `identity` 路径
        identity = x
        if self.downsample is not None:
            identity = self.downsample(x)

        # TODO: 2. 计算 Inception 分支
        out1 = self.b1(x)
        out2 = self.b2_s2(self.b2_s1(x))
        
        # TODO: 3. 拼接 Inception 输出
        out_inception = torch.cat([out1, out2], dim=1)
        
        # TODO: 4. 添加残差连接
        out = out_inception + identity
        
        # TODO: 5. 应用激活函数
        out = self.relu(out)
        
        return out


## 4. 构建你的 `MyInceptionResNet`

现在，使用你的新模块来构建一个完整的网络。

In [ ]:
class MyInceptionResNet(nn.Module):
    def __init__(self, num_classes=10):
        super(MyInceptionResNet, self).__init__()
        
        # 初始卷积层 (类似 ResNet)
        self.conv1 = BasicConv2d(3, 64, kernel_size=3, stride=1, padding=1) # 适用于 32x32 图像
        
        # TODO: 堆叠你的 InceptionResBlock 模块
        # 32x32 -> 32x32
        self.layer1 = 
        # 32x32 -> 16x16
        self.layer2 = 
        # 16x16 -> 8x8
        self.layer3 = 
        
        # TODO: 全局平均池化 (类似 GoogLeNet)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # TODO: 分类器
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# TODO: 实例化模型并测试
model = MyInceptionResNet(num_classes=10).to(device)

# 尝试用一个假数据运行，检查维度是否正确
try:
    test_input = torch.randn(2, 3, 32, 32).to(device) # 2 是 batch_size
    test_output = model(test_input)
    print(f"模型测试通过！输入 shape: {test_input.shape}, 输出 shape: {test_output.shape}")
except Exception as e:
    print(f"模型构建失败: {e}")
    raise e

## 5. 训练与评估

现在，使用标准的训练循环来训练你的新模型！

In [ ]:
# TODO: 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

save_path = './my_inception_resnet.pth'
best_acc = 0.0

print("Start Training Custom Model...")

for epoch in range(epochs):
    # --- 训练 ---
    model.train() 
    running_loss = 0.0
    train_bar = tqdm(train_loader, file=sys.stdout)
    
    for images, labels in train_bar:
        images, labels = images.to(device), labels.to(device)
        



        running_loss += loss.item()
        train_bar.desc = f"train epoch[{epoch + 1}/{epochs}] loss:{loss.item():.3f}"
    
    # --- 验证 ---
    model.eval() 
    acc = 0.0
    with torch.no_grad():
        val_bar = tqdm(test_loader, file=sys.stdout)
        for val_images, val_labels in val_bar:
            val_images, val_labels = val_images.to(device), val_labels.to(device)
            outputs = model(val_images)
            predict_y = torch.max(outputs, dim=1)[1]
            acc += torch.eq(predict_y, val_labels).sum().item()
            
    val_accurate = acc / len(test_dataset)
    print(f"[epoch {epoch + 1}] train_loss: {running_loss / len(train_loader):.3f}  val_accuracy: {val_accurate:.3f}")

    if val_accurate > best_acc:
        best_acc = val_accurate
        torch.save(model.state_dict(), save_path)
        print(f"Best model saved with accuracy: {best_acc:.3f}")

print(f"Finished Training. Best accuracy: {best_acc:.3f}")

## 6. 结论（选填）

请在此处写下你的结论或观察，例如：
* 你设计的 `InceptionResBlock` 是否成功运行了？
* 你在设计模块时遇到了哪些挑战？（例如，通道匹配或下采样）你是如何解决的？
* 你的自定义模型在 CIFAR-10 上的最终表现如何？